# Assignment 9

In [22]:
import cv2
import mediapipe as mp
import pandas as pd
import os
from mediapipe.tasks import python
from mediapipe.tasks.python import vision


def extract_mediapipe_to_csv(data_path: str, save_path: str, model_path: str):

    video_extensions = [".mp4", ".mov", ".avi", ".mkv"]
    static = not any(data_path.lower().endswith(ext) for ext in video_extensions)

    base_options = python.BaseOptions(model_asset_path=model_path)

    JOINT_ORDER = [
        "head",
        "left_shoulder", "left_elbow",
        "right_shoulder", "right_elbow",
        "left_hand", "right_hand",
        "left_hip", "right_hip",
        "left_knee", "right_knee",
        "left_foot", "right_foot"
    ]

    LANDMARK_INDEX = {
        "head": 0,  
        "left_shoulder": 11,
        "right_shoulder": 12,
        "left_elbow": 13,
        "right_elbow": 14,
        "left_hand": 15,
        "right_hand": 16,
        "left_hip": 23,
        "right_hip": 24,
        "left_knee": 25,
        "right_knee": 26,
        "left_foot": 27,
        "right_foot": 28,
    }

    data = []

    if static:
        options = vision.PoseLandmarkerOptions(
            base_options=base_options,
            running_mode=vision.RunningMode.IMAGE,
            min_pose_detection_confidence=0.5,
            min_pose_presence_confidence=0.5
        )

        with vision.PoseLandmarker.create_from_options(options) as landmarker:
            image = cv2.imread(data_path)
            if image is None:
                raise ValueError(f"Could not read image: {data_path}")

            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
            results = landmarker.detect(mp_image)

            if results.pose_landmarks:
                row = {"FrameNo": 0}

                for joint in JOINT_ORDER:
                    idx = LANDMARK_INDEX[joint]
                    lm = results.pose_landmarks[0][idx]

                    row[f"{joint}_x"] = lm.x
                    row[f"{joint}_y"] = lm.y
                    row[f"{joint}_z"] = lm.z

                data.append(row)

    else:
        options = vision.PoseLandmarkerOptions(
            base_options=base_options,
            running_mode=vision.RunningMode.VIDEO,
            min_pose_detection_confidence=0.5,
            min_pose_presence_confidence=0.5,
            min_tracking_confidence=0.5
        )

        with vision.PoseLandmarker.create_from_options(options) as landmarker:
            cap = cv2.VideoCapture(data_path)

            if not cap.isOpened():
                raise ValueError(f"Could not open video: {data_path}")

            fps = cap.get(cv2.CAP_PROP_FPS)
            frame_idx = 0

            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break

                image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)

                timestamp_ms = int((frame_idx / fps) * 1000)
                results = landmarker.detect_for_video(mp_image, timestamp_ms)

                if results.pose_landmarks:
                    row = {"FrameNo": frame_idx}

                    for joint in JOINT_ORDER:
                        idx = LANDMARK_INDEX[joint]
                        lm = results.pose_landmarks[0][idx]

                        row[f"{joint}_x"] = lm.x
                        row[f"{joint}_y"] = lm.y
                        row[f"{joint}_z"] = lm.z

                    data.append(row)

                frame_idx += 1

            cap.release()

    # ---- SAVE CSV ----
    columns = ["FrameNo"]
    for joint in JOINT_ORDER:
        columns += [f"{joint}_x", f"{joint}_y", f"{joint}_z"]

    df = pd.DataFrame(data)
    df = df[columns]

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    df.to_csv(save_path, index=False)

    print(f"Saved {len(df)} rows to {save_path}")

In [26]:
extract_mediapipe_to_csv(
    data_path="../data/1m_step_high.mov",
    save_path="../data/1m_step_high_mediapipe.csv",
    model_path="../data/pose_landmarker.task"
)

I0000 00:00:1776625494.534678 9209142 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
W0000 00:00:1776625494.664223 9341199 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776625494.692219 9341194 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Saved 397 rows to ../data/1m_step_high_mediapipe.csv


### Joint distance Enis

In [ ]:
shoulder_to_shoulder = 0.036
Shoulder_to_hip = 0.55
hip_to_hip = 0.25
hip_to_knee = 0.47
knee_to_ankle = 0.43

In [ ]:
import os

video_folder = "../../../all_videos"   
output_folder = "../../MainProject/Assignment10/data/csv_of_all_videos"        
model_path = "../data/pose_landmarker.task"

video_extensions = [".mp4", ".mov", ".avi", ".mkv"]

for file_name in os.listdir(video_folder):
    
    if not any(file_name.lower().endswith(ext) for ext in video_extensions):
        continue

    video_path = os.path.join(video_folder, file_name)

    base_name = os.path.splitext(file_name)[0]
    save_name = f"{base_name}_mediapipe.csv"
    save_path = os.path.join(output_folder, save_name)

    print(f"Processing: {file_name}")

    extract_mediapipe_to_csv(
        data_path=video_path,
        save_path=save_path,
        model_path=model_path
    )
print("Done.")

Processing: A136.avi


I0000 00:00:1776625884.231445 9209142 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
W0000 00:00:1776625884.427155 9345687 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776625884.469935 9345688 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Saved 127 rows to ../../MainProject/Assignment10/data/csv_of_all_videos/A136_mediapipe.csv
Processing: A79.avi


I0000 00:00:1776625888.168669 9209142 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
W0000 00:00:1776625888.278328 9345733 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776625888.290251 9345733 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Saved 258 rows to ../../MainProject/Assignment10/data/csv_of_all_videos/A79_mediapipe.csv
Processing: A122.avi


...

Saved 175 rows to ../../MainProject/Assignment10/data/csv_of_all_videos/A48_mediapipe.csv
Processing: A60.avi


I0000 00:00:1776626981.625376 9209142 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
W0000 00:00:1776626981.725701 9359545 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776626981.737645 9359550 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Saved 245 rows to ../../MainProject/Assignment10/data/csv_of_all_videos/A60_mediapipe.csv
Processing: A74.avi


I0000 00:00:1776626988.698186 9209142 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
W0000 00:00:1776626988.805859 9359616 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776626988.818063 9359615 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Saved 189 rows to ../../MainProject/Assignment10/data/csv_of_all_videos/A74_mediapipe.csv
Done.
